# want to test out loading my idepix data and then try to use this to mask my S2 images

In [1]:
import os
import sys
lib_path = os.path.abspath(os.path.join(os.path.abspath(''), 'functions/'))
sys.path.append(lib_path)
# need to append our functions dir to the path! 

import SD_raster_loading
import SD_NC_loading
import netCDF4
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dalecLoad
import spectralConv
import seaborn as sns
import scipy as sp
from datetime import timedelta
import matplotlib.dates as mdates

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.model_selection import GridSearchCV

In [2]:
# useful for interrogating the ncdf file (eg. to find the name we want to use for the date)
f = netCDF4.Dataset('Polymer/idepix_subset_resample_full_s2/idepix__S2A_MSIL1C_20220306T114351_N0400_R123_T30VVH_20220306T135207.nc')
# SD_spect = get_SD_NC_Spectra_grid(f, coord[0], coord[1], shape=pixel_grid_shape,
#                                   variable=variable, wavelengths=wavelengths)
f

<class 'netCDF4._netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    Conventions: CF-1.4
    TileSize: 610:610
    product_type: S2_MSI_Level-1C
    metadata_profile: beam
    metadata_version: 0.5
    start_date: 06-MAR-2022 11:43:51.023999
    stop_date: 06-MAR-2022 11:43:51.023999
    dimensions(sizes): y(75), x(99), tp_y(3), tp_x(3)
    variables(dimensions): int8 metadata(), int32 pixel_classif_flags(y, x), float32 bright_value(y, x), float32 white_value(y, x), float32 bright_white_value(y, x), float32 spectral_flatness_value(y, x), float32 ndvi_value(y, x), float32 ndsi_value(y, x), float32 radiometric_land_value(y, x), float32 radiometric_water_value(y, x), float32 b3b11_value(y, x), float32 tc1_value(y, x), float32 tc4_value(y, x), float32 tc4cirrus_water_value(y, x), float32 ndwi_value(y, x), uint16 B1(y, x), uint16 B2(y, x), uint16 B3(y, x), uint16 B4(y, x), uint16 B5(y, x), uint16 B6(y, x), uint16 B7(y, x), uint16 B8(y, x), uint16 B8A(y, x), uint16 B9(y,

In [39]:
print(f.variables['pixel_classif_flags'].flag_meanings)

IDEPIX_INVALID IDEPIX_CLOUD IDEPIX_CLOUD_AMBIGUOUS IDEPIX_CLOUD_SURE IDEPIX_CLOUD_BUFFER IDEPIX_CLOUD_SHADOW IDEPIX_SNOW_ICE IDEPIX_BRIGHT IDEPIX_WHITE IDEPIX_COASTLINE IDEPIX_LAND IDEPIX_CIRRUS_SURE IDEPIX_CIRRUS_AMBIGUOUS IDEPIX_CLEAR_LAND IDEPIX_CLEAR_WATER IDEPIX_WATER IDEPIX_BRIGHTWHITE IDEPIX_VEG_RISK IDEPIX_MOUNTAIN_SHADOW IDEPIX_POTENTIAL_SHADOW IDEPIX_CLUSTERED_CLOUD_SHADOW


In [38]:
print(f.variables['pixel_classif_flags'])

<class 'netCDF4._netCDF4.Variable'>
int32 pixel_classif_flags(y, x)
    coordinates: lat lon
    flag_meanings: IDEPIX_INVALID IDEPIX_CLOUD IDEPIX_CLOUD_AMBIGUOUS IDEPIX_CLOUD_SURE IDEPIX_CLOUD_BUFFER IDEPIX_CLOUD_SHADOW IDEPIX_SNOW_ICE IDEPIX_BRIGHT IDEPIX_WHITE IDEPIX_COASTLINE IDEPIX_LAND IDEPIX_CIRRUS_SURE IDEPIX_CIRRUS_AMBIGUOUS IDEPIX_CLEAR_LAND IDEPIX_CLEAR_WATER IDEPIX_WATER IDEPIX_BRIGHTWHITE IDEPIX_VEG_RISK IDEPIX_MOUNTAIN_SHADOW IDEPIX_POTENTIAL_SHADOW IDEPIX_CLUSTERED_CLOUD_SHADOW
    flag_masks: [      1       2       4       8      16      32      64     128     256
     512    1024    2048    4096    8192   16384   32768   65536  131072
  262144  524288 1048576]
    flag_coding_name: pixel_classif_flags
    flag_descriptions: Invalid pixels	Pixels which are either cloud_sure or cloud_ambiguous	Semi transparent clouds, or clouds where the detection level is uncertain	Fully opaque clouds with full confidence of their detection	A buffer of n pixels around a cloud. n is a us

In [37]:
print(f.variables['pixel_classif_flags'].flag_masks)

[      1       2       4       8      16      32      64     128     256
     512    1024    2048    4096    8192   16384   32768   65536  131072
  262144  524288 1048576]


In [5]:
def getclosest_ij(lats, lons, latpt, lonpt):
    '''
    a function to find the index of the point closest pt
    (in squared distance) to give lat/lon value.
    '''
    # find squared distance of every point on grid
    dist_sq = (lats-latpt)**2 + (lons-lonpt)**2
    # 1D index of minimum dist_sq element
    minindex_flattened = dist_sq.argmin()
    # Get 2D index for latvals and lonvals arrays from 1D index
    return np.unravel_index(minindex_flattened, lats.shape)

In [42]:
def get_S2_NC_IDEPIX_flag_grid(NC_file, lat_pt, lon_pt, shape=(3, 3),
                               lat_name='lat', lon_name='lon'):
    '''
    take idepix nc file and extract grid of pixels with shape=shape from it centred at lat, lon
    '''
    lat, lon = NC_file.variables[lat_name][:], NC_file.variables[lon_name][:]
    iy, ix = getclosest_ij(lat, lon, lat_pt, lon_pt)
    
                       
    # generate x and y coords for grid with shape=(shape[0], shape[1])
    x = np.linspace(ix - shape[0]//2,
                    ix + shape[0]//2 - (1 - shape[0]%2),
                    shape[0],
                    dtype=int)
    y = np.linspace(iy - shape[1]//2,
                    iy + shape[1]//2 - (1 - shape[1]%2),
                    shape[1],
                    dtype=int)
    df1 = pd.DataFrame(data={'empty?':[]})
    for i in x:
        for j in y:
            flag = ([NC_file.variables['pixel_classif_flags'][j, i]])
            # might want to think about if I want to include the lat and lon of each pixel too?
            var_name = 'IDEPIX_flag' + str(i) + '_' + str(j)
            df2 = pd.DataFrame(data={var_name:flag})
            df1 = pd.concat([df1, df2], axis=1)
    df1.drop('empty?', inplace=True, axis=1)

    return df1

In [44]:
coord = [56.147005, -3.923311]
idepix_df = get_S2_NC_IDEPIX_flag_grid(f, coord[0], coord[1])
idepix_df

,IDEPIX_flag20_29,IDEPIX_flag20_30,IDEPIX_flag20_31,IDEPIX_flag21_29,IDEPIX_flag21_30,IDEPIX_flag21_31,IDEPIX_flag22_29,IDEPIX_flag22_30,IDEPIX_flag22_31
0,148480,148480,17408,17408,17408,17408,17408,17408,17408


In [57]:
# I think for now, we'll only care about the following flags:

# IDEPIX_INVALID IDEPIX_CLOUD IDEPIX_CLOUD_AMBIGUOUS IDEPIX_CLOUD_SURE

# IDEPIX_CLOUD_BUFFER IDEPIX_CLOUD_SHADOW IDEPIX_SNOW_ICE IDEPIX_BRIGHT IDEPIX_WHITE

# so this is the first 10 flags 

# in binary that's 0b1111111111


# then add a few more which might cause issues:

# CIRRUS_SURE: 2048
# CIRRUS AMBIGIOUS: 4096
# BRIGHT_WHITE: 65536
# CLUSTERED_CLOUD_SHADOW: 1048576

flag_mask  = 0b1111111111 + 2048 + 4096 + 65536 + 1048576
flag_mask

1121279

In [58]:
idepix_df & flag_mask

,IDEPIX_flag20_29,IDEPIX_flag20_30,IDEPIX_flag20_31,IDEPIX_flag21_29,IDEPIX_flag21_30,IDEPIX_flag21_31,IDEPIX_flag22_29,IDEPIX_flag22_30,IDEPIX_flag22_31
0,False,False,False,False,False,False,False,False,False


# now just need to add in the datetime so I can match this with my satellite data

In [65]:
def load_multiple_IDEPIXs(directory, coord, pixel_grid_shape=(1, 1),
                          dateOnly=False, filetype=".nc",
                          date_name='start_date',
                          lat_name='lat', lon_name='lon'):
    '''
    load multiple idepix nc files and get grid of pixels from all of them using get_S2_NC_IDEPIX_flag_grid()
    '''
    # could use list comprehensions in a few places here if things start getting slow!
    files = []
    for file in os.listdir(directory):
        if file.endswith(filetype):
            files.append(os.path.join(directory, file))

    ncdf_dates = []
    indexes = []
    flag_list = []

    for i in range(len(files)):
        f = netCDF4.Dataset(files[i])
        flag = get_S2_NC_IDEPIX_flag_grid(f, coord[0], coord[1], shape=pixel_grid_shape,
                                              lat_name=lat_name, lon_name=lon_name)
        ncdf_dates.append(getattr(f, date_name))
        indexes.append(i)
        flag_list.append(flag)
    
    # currently my code which removes images from the same date relies on the images being sorted in date order ...
    flag_list_sorted = [x for _, x in sorted(zip(ncdf_dates, flag_list))]
    sorted_dates = sorted(ncdf_dates)
    flag_df = None
    for i in range(len(flag_list_sorted)):
        flag = flag_list_sorted[i]
        date = sorted_dates[i]

        df_tmp = flag.copy()
        df_tmp['Date'] = pd.to_datetime(date, utc=False)
        if dateOnly:
            df_tmp['Date'] = df_tmp['Date'].dt.date # just removes the time aspect from the variable
        df_tmp.set_index(['Date'], inplace=True)
        if flag_df is None:
            flag_df = df_tmp.copy()
        else:
            flag_df = pd.concat([flag_df, df_tmp])

    # don't really need to sort, but in case I change something its good to have:
    return flag_df.sort_values(['Date'])

In [68]:
idpix_flags = load_multiple_IDEPIXs('Polymer/idepix_subset_resample_full_s2/', coord, pixel_grid_shape=(3, 3))

In [103]:
df_good_data = idpix_flags & flag_mask
df_good_data = df_good_data[df_good_data==False]

In [104]:
df_good_data = df_good_data.reset_index()
df_good_data['date'] = df_good_data.Date.dt.date

In [109]:
good_dates = df_good_data.dropna().date.unique()
good_dates

array([datetime.date(2022, 3, 1), datetime.date(2022, 3, 6),
       datetime.date(2022, 3, 8), datetime.date(2022, 3, 21),
       datetime.date(2022, 3, 23), datetime.date(2022, 3, 26),
       datetime.date(2022, 3, 28), datetime.date(2022, 4, 20),
       datetime.date(2022, 4, 22), datetime.date(2022, 6, 4),
       datetime.date(2022, 7, 1), datetime.date(2022, 7, 4),
       datetime.date(2022, 7, 16), datetime.date(2022, 7, 29),
       datetime.date(2022, 8, 10), datetime.date(2022, 8, 20),
       datetime.date(2022, 9, 12), datetime.date(2022, 9, 14),
       datetime.date(2023, 3, 1), datetime.date(2023, 3, 6),
       datetime.date(2023, 3, 8), datetime.date(2023, 3, 11),
       datetime.date(2023, 4, 7), datetime.date(2023, 4, 15),
       datetime.date(2023, 4, 20), datetime.date(2023, 4, 22),
       datetime.date(2023, 5, 15), datetime.date(2023, 5, 22),
       datetime.date(2023, 5, 25), datetime.date(2023, 5, 30),
       datetime.date(2023, 6, 4), datetime.date(2023, 6, 9),
    

In [81]:
coord = [56.147005, -3.923311] # this is approx the location of the DALEC + ~10 m out which looks less edge effected 
#coord = [56.14693897799395, -3.923458784671348] # this is approx the location of the DALEC
#coord = [56.146746528609306, -3.92285731543299] # this is perhaps a deeper part of the loch
S2_polymer = SD_NC_loading.load_multiple_SDs('Polymer/polymer_S2_full_out/',
                                            coord, skipSameDay=False,
                                            pixel_grid_shape=(3,3),
                                            filetype=".nc",
                                            variable='Rw',
                                            date_name='start_time',
                                            wavelengths_S2A=[443., 492., 560., 665., 704., 740.,
                                                             783., 833., 865., 1610.],
                                            wavelengths_S2B=[442., 492., 559., 665., 704., 739.,
                                                             780., 833., 864., 1610.],
                                            div_by_pi=False,
                                            lat_name='latitude',
                                            lon_name='longitude')

are you sure you know the wavelength order? - best double check it!
Rw443 : 443.0
Rw490 : 492.0
Rw560 : 560.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 740.0
Rw783 : 783.0
Rw842 : 833.0
Rw865 : 865.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 

are you sure you know the wavelength order? - best double check it!
Rw443 : 443.0
Rw490 : 492.0
Rw560 : 560.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 740.0
Rw783 : 783.0
Rw842 : 833.0
Rw865 : 865.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 443.0
Rw490 : 492.0
Rw560 : 560.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 740.0
Rw783 : 783.0
Rw842 : 833.0
Rw865 : 865.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 443.0
Rw490 : 492.0
Rw560 : 560.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 740.0
Rw783 : 

are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 443.0
Rw490 : 492.0
Rw560 : 560.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 740.0
Rw783 : 783.0
Rw842 : 833.0
Rw865 : 865.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 442.0
Rw490 : 492.0
Rw560 : 559.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 739.0
Rw783 : 780.0
Rw842 : 833.0
Rw865 : 864.0
Rw1610 : 1610.0
are you sure you know the wavelength order? - best double check it!
Rw443 : 443.0
Rw490 : 492.0
Rw560 : 560.0
Rw665 : 665.0
Rw705 : 704.0
Rw740 : 740.0
Rw783 : 

In [93]:
S2_polymer = S2_polymer.reset_index()
S2_polymer['date'] = S2_polymer.Date.dt.date

In [114]:
good_polymer = S2_polymer[S2_polymer.date.isin(good_dates)]
good_polymer

,Date,Wavelength,rho_s_20_29,rho_s_20_30,rho_s_20_31,rho_s_21_29,rho_s_21_30,rho_s_21_31,rho_s_22_29,rho_s_22_30,rho_s_22_31,date
0,2022-03-01 11:43:49,442.0,0.017567342,0.010720307,0.010181804,0.008952142,0.007511505,0.009897182,0.008275511,0.0097508,0.011342642,2022-03-01
1,2022-03-01 11:43:49,492.0,0.015728079,0.010308605,0.010381798,0.010111533,0.008939228,0.011184513,0.009043262,0.010714174,0.01252463,2022-03-01
2,2022-03-01 11:43:49,559.0,0.012375816,0.0104213515,0.0072902283,0.010377663,0.009635738,0.010463015,0.012231069,0.014829234,0.01324929,2022-03-01
3,2022-03-01 11:43:49,665.0,-0.003951123,-0.0028254832,-0.0003339649,0.0006797325,0.0010128936,0.00055551756,0.0008127278,0.000996801,0.00025838584,2022-03-01
4,2022-03-01 11:43:49,704.0,0.0019598177,0.0020084514,0.00074186653,0.0004055274,0.0007643812,0.00063734653,0.0013473639,0.0019020245,0.0010360131,2022-03-01
...,...,...,...,...,...,...,...,...,...,...,...,...
1065,2023-10-22 11:43:49,739.0,0.007963073,0.0069648987,0.006782831,0.0073719304,0.0036469076,0.0024231751,0.0039417874,0.0020105576,0.0017770302,2023-10-22
1066,2023-10-22 11:43:49,780.0,0.005828218,0.003778014,0.0037656645,0.0041456847,0.0039623496,0.0045240326,0.0034446637,0.004216749,0.00453969,2023-10-22
1067,2023-10-22 11:43:49,833.0,-0.042024508,-0.023085624,-0.01647179,-0.007004199,-0.0032959976,0.00016633082,-0.0001573608,0.00035914898,0.0020039442,2023-10-22
1068,2023-10-22 11:43:49,864.0,-0.0078414725,-0.0040515703,-0.0038457643,-0.004686667,-0.0031554154,-0.0029191459,-0.0030402467,-0.0025852327,-0.0026274237,2023-10-22


In [115]:
good_polymer.info()

<class 'pandas.core.frame.DataFrame'>
Index: 470 entries, 0 to 1069
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         470 non-null    datetime64[ns]
 1   Wavelength   470 non-null    float64       
 2   rho_s_20_29  470 non-null    object        
 3   rho_s_20_30  470 non-null    object        
 4   rho_s_20_31  470 non-null    object        
 5   rho_s_21_29  470 non-null    object        
 6   rho_s_21_30  470 non-null    object        
 7   rho_s_21_31  470 non-null    object        
 8   rho_s_22_29  470 non-null    object        
 9   rho_s_22_30  470 non-null    object        
 10  rho_s_22_31  470 non-null    object        
 11  date         470 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(10)
memory usage: 47.7+ KB


# Nice. Now can incorporate into my S2 atmospheric correction code.